# FINC 430 Project Portfolio — Component 2: Exploratory Data Analysis
## JKP U.S. factor *themes*: 13 monthly long–short return series, 1926–2025

**What this notebook does.** Before we fit any model (Component 3 uses KNN regression), we need to know what is *in* the data: how it is organized, where it is trustworthy, how risky and how correlated the series are, whether the story is stable over time, and whether returns show any month-to-month persistence that a forecasting model could use.

**Roadmap**

1. Load the data and take a first look
2. Integrity and coverage checks
3. A surprise: `n_factors` and the very noisy early years → choosing the analysis sample
4. Reshaping long → wide (the same `dcast` idea as in the class JKP notebook)
5. Summary statistics: return, risk, Sharpe-like ratio, tails
6. Pictures: cumulative returns, distributions, outliers
7. Correlations: are 13 themes really 13 bets?
8. Stability over time: decades, rolling windows, the value theme's "winter"
9. Persistence: autocorrelation (the bridge to KNN)
10. Takeaways

**How to run (Colab, R runtime).** Upload `[usa]_[all_themes]_[monthly]_[vw_cap].csv` (or the `.zip` that contains it) to the session's file pane, then run the cells top to bottom. Only `data.table` and `ggplot2` are needed.

In [ ]:
# Install any missing packages (first run on Colab can take a few minutes)
for (p in c("data.table", "ggplot2")) if (!requireNamespace(p, quietly = TRUE)) install.packages(p)
library(data.table)
library(ggplot2)

options(repr.plot.width = 10, repr.plot.height = 6)   # default plot size in Jupyter/Colab
theme_set(theme_minimal(base_size = 12))


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%




## 1. Load the data and take a first look

The file holds the **theme-level** monthly returns for U.S. stocks from the Jensen–Kelly–Pedersen (JKP) global factor database. Each theme bundles several closely related factors (for example, the *value* theme averages book-to-market, earnings yield, and other cheapness measures). Weighting is "vw_cap": value-weighted with caps, so the very largest stocks cannot dominate and micro-caps matter little.

In [ ]:
find_file <- function(pat) list.files(pattern = pat)[1]
data_file <- find_file("all_themes.*\\.csv$")
if (is.na(data_file)) {                       # maybe only the zip was uploaded
  z <- find_file("all_themes.*\\.zip$")
  if (!is.na(z)) { unzip(z); data_file <- find_file("all_themes.*\\.csv$") }
}
if (is.na(data_file)) stop("Upload the JKP themes CSV (or its zip) to the working folder first.")

raw <- fread(data_file)
raw[, date := as.Date(date)]

dim(raw)
str(raw)
head(raw)
tail(raw)
raw[, lapply(.SD, uniqueN), .SDcols = c("location", "name", "freq", "weighting")]

ERROR: Error: Upload the JKP themes CSV (or its zip) to the working folder first.


**Reading the structure.** The data are in *long* format: one row per (theme, month). `name` is the theme, `date` is the month-end, `ret` is that month's return of the theme's long–short portfolio (a zero-investment portfolio: long the stocks that rank high on the theme's characteristics, short the stocks that rank low, signed so that a positive number is the direction the academic literature expects to be rewarded). `location`, `freq`, and `weighting` are constants here (USA / monthly / vw_cap) — they simply label the file. `n_factors` is the number of underlying factors averaged into that theme *in that month* — it turns out to matter a lot (Section 3).

## 2. Integrity and coverage checks

In [ ]:
cat("Duplicate (theme, month) rows:", anyDuplicated(raw[, .(name, date)]), "\n")
cat("Missing returns:              ", sum(is.na(raw$ret)), "\n\n")

coverage <- raw[order(date), .(first_month = min(date), last_month = max(date), n_months = .N,
                               nf_first = n_factors[1], nf_last = n_factors[.N], nf_median = as.numeric(median(n_factors))),
                by = name][order(first_month)]
coverage

No duplicates and no missing values — but the series do **not** all start at the same time. Most themes begin in 1926; `investment` in 1931, `accruals` in 1946, and `debt_issuance` only in November 1951 (accounting-based variables need firm financial statements, which are available in electronic form much later than prices). So the full data set is an *unbalanced panel*: 14,945 theme-months rather than 13 × 1,200.

## 3. A surprise: `n_factors` and the noisy early years

`n_factors` is not constant. In the 1920s–40s a theme such as *value* is built from only 1–3 factors; by the 1970s it is built from 18. Fewer ingredients mean a noisier portfolio. Let's look.

In [ ]:
ggplot(raw, aes(date, n_factors)) +
  geom_step(color = "#1f4e79") +
  facet_wrap(~ name, scales = "free_y", ncol = 4) +
  labs(title = "How many underlying factors feed each theme, month by month",
       x = NULL, y = "n_factors")

In [ ]:
# The ten most extreme theme-months in the whole file
raw[order(-abs(ret))][1:10, .(name, month = format(date, "%Y-%m"), ret_pct = round(100 * ret, 1), n_factors)]

In [ ]:
# Pooled volatility of monthly theme returns by decade, next to the average number of factors behind each theme
raw[, decade := paste0(floor(year(date) / 10) * 10, "s")]
raw[, .(pooled_sd_pct = round(100 * sd(ret), 2), avg_n_factors = round(mean(n_factors), 1), n_obs = .N),
    by = decade][order(decade)]

**What this tells us.** All ten most extreme observations come from 1932–1933 and 1939 (the Great Depression and the outbreak of war) when themes rested on only 1–6 factors; e.g., *quality* lost 59% in August 1932 using 2 factors. Pooled volatility in the 1930s is about 6.3% per month versus roughly 1–2% in the 1950s–80s. That is partly a genuinely violent market, and partly a thin, noisy construction.

**Decision (analysis sample).** We use **January 1952 – December 2025**. By then all 13 themes exist (the last one, `debt_issuance`, starts in Nov 1951) and each theme is built from a mature set of factors. This also gives us a balanced panel, which we need for cross-theme comparisons and for the KNN features in Component 3. The cost: we discard 25 years of history, including the 1929–33 crash. We think a clean 74-year sample beats a longer but patchy one — and a professor/reader can see exactly why.

In [ ]:
wide_all <- dcast(raw, date ~ name, value.var = "ret")          # long -> wide: one column per theme
cat("First month with all 13 themes present:", format(wide_all[complete.cases(wide_all), min(date)]), "\n")

start_date <- as.Date("1952-01-01")
an   <- raw[date >= start_date]                                   # analysis sample, still long
wide <- dcast(an, date ~ name, value.var = "ret")                 # analysis sample, wide
stopifnot(!anyNA(wide))
theme_cols <- setdiff(names(wide), "date")

cat("Analysis sample:", nrow(wide), "months x", length(theme_cols), "themes =", nrow(an), "theme-months\n")
cat("From", format(min(wide$date)), "to", format(max(wide$date)), "\n")
head(wide)

## 4. Long vs. wide
We keep both versions on purpose. **Long** (`an`) is best for grouped summaries and for `ggplot2` facets (`by = name`). **Wide** (`wide`) is best for correlations and for building features that use several themes at once. `dcast()` goes long → wide; `melt()` goes back.

## 5. Summary statistics
Returns are monthly decimals (0.01 = 1%). Because the portfolios are long–short (zero net investment), there is no risk-free rate to subtract; the "Sharpe" below is simply annualized mean ÷ annualized standard deviation. The *t-stat* tests whether the mean differs from zero (assuming independent months, so treat it as a rough guide).

In [ ]:
skew   <- function(x) mean((x - mean(x))^3) / sd(x)^3
exkurt <- function(x) mean((x - mean(x))^4) / sd(x)^4 - 3        # 0 for a normal distribution

stats <- an[, .(ann_mean_pct = 1200 * mean(ret),
                ann_vol_pct  = 100 * sqrt(12) * sd(ret),
                sharpe       = sqrt(12) * mean(ret) / sd(ret),
                t_stat       = mean(ret) / (sd(ret) / sqrt(.N)),
                skew         = skew(ret),
                ex_kurtosis  = exkurt(ret),
                pct_months_up = 100 * mean(ret > 0),
                worst_pct    = 100 * min(ret),
                worst_month  = format(date[which.min(ret)], "%Y-%m"),
                best_pct     = 100 * max(ret)),
            by = name][order(-sharpe)]
num_cols <- names(stats)[sapply(stats, is.numeric)]
stats[, (num_cols) := lapply(.SD, round, 2), .SDcols = num_cols]
stats

**Reading the table.** Premiums are modest: annualized means range from about 0% to 4.8%, against annualized volatilities of 2–12%. The best risk-adjusted series are the quiet ones (*debt_issuance*, *seasonality*, *accruals*, *quality*), while the headline-famous themes are more volatile and less reliable: *value* has a Sharpe of 0.35 and *momentum* 0.46. Three themes are statistically indistinguishable from zero over 1952–2025: *size*, *low_risk*, and especially *low_leverage* (annualized mean of 0.08%). Almost every theme has fat tails (excess kurtosis far above 0), and the worst months cluster in time — look at the `worst_month` column.

## 6. Pictures: cumulative returns, distributions, and outliers

In [ ]:
setorder(an, name, date)
an[, cum_log_ret := cumsum(log1p(ret)), by = name]

ggplot(an, aes(date, cum_log_ret)) +
  geom_hline(yintercept = 0, linewidth = 0.3, color = "grey40") +
  geom_line(color = "#1f4e79") +
  facet_wrap(~ name, ncol = 4, scales = "free_y") +
  labs(title = "Cumulative log return of each theme, 1952–2025",
       subtitle = "Note the free y-axes: slopes are comparable within a panel, levels are not across panels",
       x = NULL, y = "Cumulative log return")

In [ ]:
ord <- stats[order(ann_vol_pct), name]
an[, name_f := factor(name, levels = ord)]

ggplot(an, aes(name_f, ret)) +
  geom_boxplot(fill = "#cfe2f3", outlier.size = 0.8, outlier.alpha = 0.4) +
  coord_flip() +
  labs(title = "Distribution of monthly returns by theme (ordered by volatility)",
       x = NULL, y = "Monthly return")

In [ ]:
v <- an[name == "value", ret]
ggplot(data.frame(ret = v), aes(ret)) +
  geom_histogram(aes(y = after_stat(density)), bins = 60, fill = "#9fc5e8", color = "white") +
  stat_function(fun = dnorm, args = list(mean = mean(v), sd = sd(v)), color = "#cc0000", linewidth = 1) +
  labs(title = "Value theme: monthly returns vs. a normal curve with the same mean and SD",
       subtitle = "Peaked in the middle, heavy in the tails",
       x = "Monthly return", y = "Density")

In [ ]:
# The 12 worst theme-months in the analysis sample
an[order(ret)][1:12, .(name, month = format(date, "%Y-%m"), ret_pct = round(100 * ret, 1))]

In [ ]:
# The outlier rule used in the class JKP notebook (1.5 x IQR fences), applied here only to COUNT outliers
remove_outliers <- function(x, na.rm = TRUE, ...) {
  qnt <- quantile(x, probs = c(.25, .75), na.rm = na.rm, ...)
  H <- 1.5 * IQR(x, na.rm = na.rm)
  y <- x
  y[x < (qnt[1] - H)] <- NA
  y[x > (qnt[2] + H)] <- NA
  y
}
an[, .(pct_months_flagged = round(100 * mean(is.na(remove_outliers(ret))), 1)), by = name][order(-pct_months_flagged)]

**Outliers: keep them.** The class notebook provides a function that blanks out returns beyond 1.5 IQRs. Applied here it flags roughly 3–7% of months per theme — far more than a normal distribution would (about 0.7%). But these are not data errors. Ten of the 12 worst theme-months fall between December 1999 and February 2001, the peak and bust of the dot-com bubble; **February 2000** alone was the single worst month for *value*, *low_risk*, *investment*, and *profitability*, as growth stocks surged and cheap, safe, and profitable stocks lagged. The two exceptions are *momentum* (April 2009 and January 1975, both sharp rebounds of previously beaten-down stocks). Deleting them would delete exactly the risk an investor in these strategies is paid to bear. So we do **not** drop outliers; instead we remember them when we choose a model and a loss function in Component 3 (squared-error loss is dominated by such months).

## 7. Correlations: are 13 themes really 13 bets?
We order the themes by hierarchical clustering so that themes that move together sit next to each other.

In [ ]:
cm  <- cor(wide[, ..theme_cols])
ord <- hclust(as.dist(1 - cm))$order
cm_long <- as.data.table(as.table(cm[ord, ord]))                  # columns: V1, V2, N
cm_long[, `:=`(V1 = factor(V1, levels = rownames(cm)[ord]), V2 = factor(V2, levels = rownames(cm)[ord]))]

ggplot(cm_long, aes(V1, V2, fill = N)) +
  geom_tile() +
  geom_text(aes(label = sprintf("%.2f", N)), size = 2.7) +
  scale_fill_gradient2(low = "#b2182b", mid = "white", high = "#2166ac", limits = c(-1, 1), name = "corr") +
  coord_fixed() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Correlation of monthly theme returns, 1952–2025", x = NULL, y = NULL)

In [ ]:
od <- cm[upper.tri(cm)]
round(c(mean = mean(od), min = min(od), max = max(od)), 2)

pr <- as.data.table(as.table(cm))
pr[, `:=`(a = as.character(V1), b = as.character(V2))]
pr[a < b][order(-abs(N))][1:8, .(theme_a = a, theme_b = b, corr = round(N, 2))]

**Takeaway.** The *average* pairwise correlation is about zero — so a basket of themes diversifies well — but the average hides strong structure. *Value*, *investment*, and *low_risk* move together (correlations of roughly +0.7 to +0.8), while *low_leverage* is almost a mirror image of *value* (−0.91). That makes economic sense: high-leverage firms tend to be "cheap" firms, so a low-leverage long–short portfolio ends up short value. The practical lesson for modeling: themes are not interchangeable inputs, and a feature that averages across themes (as we do in Component 3) mixes very different exposures.

## 8. Stability over time
Is a theme's average return a constant of nature, or does it drift? Two views: Sharpe by decade, and rolling five-year average returns.

In [ ]:
an[, decade := paste0(floor(year(date) / 10) * 10, "s")]
dec <- an[, .(sharpe = sqrt(12) * mean(ret) / sd(ret)), by = .(name, decade)]

ggplot(dec, aes(decade, factor(name, levels = rev(sort(unique(name)))), fill = sharpe)) +
  geom_tile(color = "white") +
  geom_text(aes(label = sprintf("%.1f", sharpe)), size = 3) +
  scale_fill_gradient2(low = "#b2182b", mid = "white", high = "#2166ac", midpoint = 0, name = "Sharpe") +
  labs(title = "Annualized Sharpe-like ratio by decade", subtitle = "2020s = 2020–2025 only", x = NULL, y = NULL)

In [ ]:
roll <- an[name %in% c("value", "momentum", "quality", "low_risk")]
roll[, roll60 := 1200 * frollmean(ret, 60, align = "right"), by = name]

ggplot(roll, aes(date, roll60, color = name)) +
  geom_hline(yintercept = 0, linetype = "dashed") +
  geom_line(na.rm = TRUE) +
  labs(title = "Rolling 5-year average return (annualized %), four well-known themes",
       x = NULL, y = "Annualized %", color = NULL)

In [ ]:
# The value theme in four eras
vp <- an[name == "value",
         .(months = .N,
           ann_mean_pct = round(1200 * mean(ret), 1),
           ann_vol_pct  = round(100 * sqrt(12) * sd(ret), 1),
           sharpe       = round(sqrt(12) * mean(ret) / sd(ret), 2)),
         by = .(period = cut(date, as.Date(c("1952-01-01", "1980-01-01", "2007-01-01", "2018-01-01", "2026-01-01")),
                             labels = c("1952-79", "1980-2006", "2007-17", "2018-25"), right = FALSE))]
vp

**Takeaway.** Nothing here is constant. *Value* earned a Sharpe near 0.8 in the 1970s and 1980s, about −0.15 in the 1990s (the dot-com boom), recovered in the 2000s, and has been weak since 2007 — its annualized average return fell from about 3–6% in the first two eras to roughly 1.5% in the last two. Many themes show weaker numbers after 2000 (e.g., *accruals*, *short_term_reversal*, *seasonality*), consistent with the idea that returns shrink after a pattern becomes well known and widely traded. For prediction this is a warning: a model trained on the past is learning from a population of months that may not look like the future.

## 9. Persistence: does last month tell us anything about next month?
KNN will predict next month's return from *past* information. If returns had no persistence at all, past returns could not help. The lag-1 autocorrelation (correlation between this month's and next month's return) is the simplest check. We also compute it for absolute returns, a rough gauge of *volatility clustering* (calm months follow calm months).

In [ ]:
lag1 <- function(x) acf(x, lag.max = 1, plot = FALSE)$acf[2]
ac <- data.table(theme  = theme_cols,
                 ac_ret = sapply(theme_cols, function(t) lag1(wide[[t]])),
                 ac_abs = sapply(theme_cols, function(t) lag1(abs(wide[[t]]))))
band <- 2 / sqrt(nrow(wide))
cat("Rough 95% noise band for a zero autocorrelation: +/-", round(band, 3), "\n")

ggplot(ac, aes(reorder(theme, ac_ret), ac_ret)) +
  geom_col(fill = "#1f4e79") +
  geom_hline(yintercept = c(-band, band), linetype = "dashed", color = "#cc0000") +
  coord_flip() +
  labs(title = "Lag-1 autocorrelation of monthly theme returns",
       subtitle = "Dashed lines: approximate 95% band under 'no autocorrelation'", x = NULL, y = "Autocorrelation")

ac[order(-ac_ret)][, `:=`(ac_ret = round(ac_ret, 3), ac_abs = round(ac_abs, 3))][]

In [ ]:
# Value theme: this month's return vs. next month's return
v_pairs <- data.table(r_t = head(wide$value, -1), r_next = tail(wide$value, -1))
ggplot(v_pairs, aes(r_t, r_next)) +
  geom_point(alpha = 0.3) +
  geom_smooth(method = "lm", formula = y ~ x, color = "#cc0000") +
  labs(title = "Value theme: return this month vs. next month",
       x = "Return in month t", y = "Return in month t+1")

**Takeaway.** Ten of the 13 themes have *positive* lag-1 autocorrelation at or above the noise band (the last few only just) — about 0.14–0.16 for *accruals*, *profitability*, *investment*, and *value* — while *short_term_reversal* is slightly negative (−0.08) and *momentum* is near zero. Absolute returns are even more persistent for several themes (volatility clustering). So there is *some* month-to-month structure to learn from, but it is small: an autocorrelation of 0.14 means last month explains roughly 2% of the variance of this month. The scatter plot shows it: a visible tilt in the fitted line, buried inside a huge cloud. Persistence like this can also come from mechanical sources (overlapping formation windows, slowly updated accounting data), so it is a clue for a forecaster, not proof of a tradable edge.

*Caution:* everything in this notebook uses the full 1952–2025 sample, which is fine for describing the data. In Component 3 we respect the forecasting setup strictly: tune and fit on the early part and test on the later part.

## 10. Takeaways

1. **Structure.** 13 themes × monthly returns; long format; clean (no duplicates or NAs) but unbalanced. The analysis sample is Jan 1952 – Dec 2025: 888 months × 13 themes = 11,544 observations.
2. **Data quality is time-varying.** `n_factors` grows from 1–6 early on to as many as 22 later; the early years are noisy and produce the most extreme outliers (1932–33, 1939). We start in 1952 on purpose.
3. **Modest premiums, fat tails.** Annualized average returns are roughly 0–5% with annualized Sharpe-like ratios mostly between 0.1 and 1.0. Value (0.35) and momentum (0.46) are famous but not the steadiest; *low_leverage* is statistically indistinguishable from zero. Tail risk is large and clustered: the 1999–2001 dot-com episode above all, with 2009 and 2020 also producing some themes' worst months.
4. **Diversification with hidden clusters.** The average pairwise correlation is about zero, but value / investment / low_risk move together and low_leverage is value's mirror image (−0.91).
5. **Regimes.** Premiums are not stable: value was strong in the 1970s–80s, negative in the 1990s, and weak after 2007.
6. **Persistence exists but is small.** Lag-1 autocorrelations of 0.1–0.16 for many themes, plus volatility clustering.

**What this means for Component 3 (KNN regression).** We will predict next month's return of one theme (value) from (i) its own recent return, average, and volatility and (ii) the cross-theme average return. We expect KNN to find little, because the monthly signal-to-noise ratio is low (monthly standard deviation ≈ 3%, monthly mean ≈ 0.3%), and we will need to (a) split the data chronologically rather than randomly, (b) standardize features, and (c) interpret results against a naive benchmark rather than in isolation.